# Testing Pruning and Anti-Pruning

I need to see how the ESSP and HSP are affected when I trim the prefix down to just the high-importance sentences.

To start, I'll set up the infrastructure to test that the baseline ESSP and HSP step-accuracies I have are correct.

## Load Problem 1 from the Dataset

In [1]:
import json

dataset = []
with open("../data/hsp_step_accuracies.jsonl", 'r') as f:
    for line in f:
        dataset.append(json.loads(line))

problem1 = dataset[0]

print(problem1.keys())

essp_step_accuracies = problem1['essp_step_accuracies'][:-1]
hsp_step_accuracies = problem1['hsp_step_accuracies']

essp_index = problem1['first_essp_index']
# A nuance of running the hsp_step_accuracy finding process
# after the initial hsp finding process.
# There's a disconnect between the hsp_index and its corresponding
# accuracy.
# To get arround this, the most principles thing to do would be to
# run the model at both points, average accuracy across 20 rollouts
# and use whichever step is above the threshold, and earlier.
hsp_index1 = problem1['first_hsp_index']
hsp_index2 = 335

cot_up_to_essp = "\n".join(problem1['steps'][:essp_index+1])

cot_up_to_hsp1 = "\n".join(problem1['steps'][:hsp_index1+1])
cot_up_to_hsp2 = "\n".join(problem1['steps'][:hsp_index2+1])

dict_keys(['id', 'problem', 'solution', 'full_cot', 'steps', 'total_steps', 'first_essp_index', 'essp_step_accuracies', 'essp_indices', 'answer', 'first_hsp_index', 'hsp_step_accuracies'])


In [2]:
len(problem1['steps'])

181

Now that I have the problem, and all the requisite variables, let's start by testing that the ESSP accuracy holds. This means running the model on `cot_up_to_essp` and forcing it to answer early.

## Load the vLLM model and tokenizer

In [3]:
import os
from vllm import LLM, SamplingParams

os.environ["CUDA_VISIBLE_DEVICES"] = "2"
    
# Load the model
model = LLM(
    model="Qwen/Qwen3-4B",
    seed=9001,
    trust_remote_code=True,
    gpu_memory_utilization=0.95
)
tokenizer = model.get_tokenizer()

/disk/u/gio/.conda/envs/transfer/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO 12-21 14:00:08 [utils.py:253] non-default args: {'trust_remote_code': True, 'seed': 9001, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 12-21 14:00:14 [model.py:514] Resolved architecture: Qwen3ForCausalLM
INFO 12-21 14:00:14 [model.py:1661] Using max model len 40960


2025-12-21 14:00:14,128	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 12-21 14:00:14 [scheduler.py:230] Chunked prefill is enabled with max_num_batched_tokens=8192.
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:14 [core.py:93] Initializing a V1 LLM engine (v0.13.0) with config: model='Qwen/Qwen3-4B', speculative_config=None, tokenizer='Qwen/Qwen3-4B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=40960, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_fallback=False, disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_parser_plugin='', enable_in_reasoning=False), observability_config=ObservabilityConfig(show_hidden_metrics_for_version=None, otlp_traces_endpoint=N

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:  33% Completed | 1/3 [00:00<00:01,  1.58it/s]
Loading safetensors checkpoint shards:  67% Completed | 2/3 [00:00<00:00,  2.92it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.26it/s]
Loading safetensors checkpoint shards: 100% Completed | 3/3 [00:01<00:00,  2.25it/s]
(EngineCore_DP0 pid=3756607) 


(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:18 [default_loader.py:308] Loading weights took 1.44 seconds
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:18 [gpu_model_runner.py:3659] Model loading took 7.5552 GiB memory and 1.906409 seconds
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:24 [backends.py:643] Using cache directory: /disk/u/gio/.cache/vllm/torch_compile_cache/3fa0a64d2d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:24 [backends.py:703] Dynamo bytecode transform time: 6.01 s
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:29 [backends.py:261] Cache the graph of compile range (1, 8192) for later use
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:35 [backends.py:278] Compiling a graph for compile range (1, 8192) takes 7.60 s
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:35 [monitor.py:34] torch.compile takes 13.61 s in total
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:36 [gpu_worker.py:375] Available KV cache memory: 66.27 GiB
(Engine

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:01<00:00, 30.83it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:00<00:00, 38.02it/s]


(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:39 [gpu_model_runner.py:4587] Graph capturing finished in 3 secs, took 0.60 GiB
(EngineCore_DP0 pid=3756607) INFO 12-21 14:00:39 [core.py:259] init engine (profile, create kv cache, warmup model) took 20.98 seconds
INFO 12-21 14:00:40 [llm.py:360] Supported tasks: ['generate']


In [4]:
params_handoff = SamplingParams(
    n=20,
    temperature=0.7,
    top_p=0.95,
    max_tokens=31_000,
    seed=9001
)

In [5]:
cot_up_to_hsp2

"Okay, so I need to figure out how many endpoints Figure 5 will have if we continue the pattern shown.\nLet me start by understanding the pattern from the given figures.\nFirst, let me look at Figure 1.\nIt's a single line segment with three segments coming out of the top, forming a Y shape.\nWait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).\nSo, it's like a central point with three lines coming out: one straight down, and two going up to the sides.\nSo, the endpoints here would be the tips of these three lines.\nThe bottom tip is at (0,-3), and the two upper tips are at (-2,2) and (2,2).\nSo, Figure 1 has 3 endpoints.\nNow, moving to Figure 2.\nThe Asymptote code draws a more complex figure.\nLet me try to visualize it.\nThe description says each line-segment extremity is replaced by a gradually smaller Y.\nSo, in Figure 1, each endpoint is replaced by a Y in Figure 2.\nLe

## Build the Prompt

In [ ]:
problem_prompt = problem1['problem']

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

base_prompt = build_base_prompt(problem_prompt)

prompt = f"{base_prompt}<think>\n{cot_up_to_hsp2}"
#prompt += "\n</think>\nTherefore, the final answer is \\boxed{"
prompt


'<|im_start|>system\nYou are a helpful reasoning assistant. Output your final answer inside \\boxed{}.<|im_end|>\n<|im_start|>user\nIf you continue this pattern in which each line-segment extremity is replaced by a gradually smaller Y in the next figure, in the manner shown, how many endpoints will Figure 5 have?\n\n[asy]\ndraw((0,0)--(0,-3),linewidth(.75));\ndraw((0,0)--(-2,2),linewidth(.75));\ndraw((0,0)--(2,2),linewidth(.75));\nlabel("Figure 1",(0,-3),S);\n\ndraw((5,0)--(5,-2),linewidth(.75));\ndraw((4,-3)--(5,-2)--(6,-3),linewidth(.75));\ndraw((4,1)--(5,0)--(6,1),linewidth(.75));\ndraw((3,1)--(4,1)--(4,2),linewidth(.75));\ndraw((6,2)--(6,1)--(7,1),linewidth(.75));\nlabel("Figure 2",(5,-3),S);\n\ndraw((10,0)--(10,-2),linewidth(.75));\ndraw((9.5,-2.5)--(10,-2)--(10.5,-2.5),linewidth(.75));\ndraw((9,-2.5)--(9.5,-2.5)--(9.5,-3),linewidth(.75));\ndraw((11,-2.5)--(10.5,-2.5)--(10.5,-3),linewidth(.75));\n\ndraw((9,1)--(10,0)--(11,1),linewidth(.75));\ndraw((8.5,1)--(9,1)--(9,1.5),linewidth

## Run the Model

In [ ]:
outputs = model.generate(
    [prompt], params_handoff, use_tqdm=True
)

Processed prompts:   0%|          | 0/20 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

In [ ]:
# Get generation results
output = outputs[0]
completions = [o.text for o in output.outputs]
print(completions)

[' Then from (4,1) goes to (3,1) and up to (4,2). Similarly, from (6,1) goes to (7,1) and up to (6,2).\nThen from (3,1) goes to (something)? Wait, the Asymptote code draws (3,1)--(4,1)--(4,2). So (3,1) is an endpoint? And (4,2) is an endpoint? Similarly, on the right side, (7,1) and (6,2) would be endpoints. Then there are more lines: (8.5,1)--(9,1)--(9,1.5)... Wait, no, that\'s for Figure 3. Let me focus on Figure 2.\n\nSo, Figure 2: bottom endpoints at (4,-3) and (6,-3). Then middle part: from (5,0) up to (5,-2)? Wait, no. Let me parse again.\n\nOriginal lines for Figure 2:\n\ndraw((5,0)--(5,-2),linewidth(.75));\ndraw((4,-3)--(5,-2)--(6,-3),linewidth(.75));\nSo, starting from (5,0) going down to (5,-2). Then from (5,-2), connecting to (4,-3) and (6,-3). So (4,-3) and (6,-3) are endpoints here.\n\nThen:\n\ndraw((4,1)--(5,0)--(6,1),linewidth(.75));\n\nSo, from (5,0) going up to (4,1) and (6,1). So (4,1) and (6,1) are connected to (5,0). Then:\n\ndraw((3,1)--(4,1)--(4,2),linewidth(.75))

In [ ]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

ground_truth = problem1['solution']

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'


1.0

## Testing Pruned HSP

Now I need to identify which sentences are selected to be removed, remove them, and rerun the model on this pruned CoT and evaluate.

In [7]:
# Load the attribtion data
with open('../data/attributions/result1.1.json', 'r') as f:
    attr_result = json.load(f)

In [23]:
import numpy as np
import re
from typing import Dict, List, Tuple, Optional

def segment_tokens_with_cot_splitter(result: dict, include_prompt_segments: bool = True):
    """
    Applies CoTSplitter logic to tokens, with proper handling of special tokens
    and prompt structure.
    
    Args:
        result: Dict with 'tokens' and 'attributions' keys
        include_prompt_segments: If True, include system/user prompt as segments
    
    Returns:
        List of segment dictionaries with step_text, tokens, strength, etc.
    """
    tokens = result["tokens"]
    attributions = result["attributions"]
    
    # Special token patterns (adjust based on your tokenizer)
    SPECIAL_TOKENS = {
        '<|im_start|>', '<|im_end|>', '<think>', '</think>', 
        '<|im_start|>system', '<|im_start|>user', '<|im_start|>assistant'
    }
    
    def decode_token(tok):
        return tok.replace('Ġ', ' ').replace('Ċ', '\n')
    
    def is_special_token(tok):
        # Check if token is or starts a special sequence
        return tok in SPECIAL_TOKENS or tok.startswith('<|') or tok in ['<think>', '</think>']
    
    def find_token_index(tokens, target, start=0):
        """Find index of target token starting from start position."""
        for i in range(start, len(tokens)):
            if tokens[i] == target or target in tokens[i]:
                return i
        return -1
    
    def find_sequence(tokens, sequence, start=0):
        """Find starting index of a token sequence."""
        for i in range(start, len(tokens) - len(sequence) + 1):
            if all(tokens[i+j] == sequence[j] for j in range(len(sequence))):
                return i
        return -1
    
    # Identify structural boundaries
    boundaries = []
    
    # Find im_start and im_end positions
    im_start_positions = [i for i, t in enumerate(tokens) if '<|im_start|>' in t]
    im_end_positions = [i for i, t in enumerate(tokens) if '<|im_end|>' in t]
    think_start = find_token_index(tokens, '<think>')
    think_end = find_token_index(tokens, '</think>')
    
    # Determine prompt boundaries
    # Typically: <|im_start|>system....<|im_end|>\n<|im_start|>user....<|im_end|>\n<|im_start|>assistant\n<think>
    
    segment_data = []
    
    if include_prompt_segments and len(im_start_positions) >= 2:
        # System prompt segment
        if len(im_end_positions) >= 1:
            sys_start = im_start_positions[0]
            sys_end = im_end_positions[0]
            segment_data.append(create_segment(
                tokens, attributions, sys_start, sys_end + 1,
                segment_type='system_prompt'
            ))
        
        # User prompt segment  
        if len(im_end_positions) >= 2:
            user_start = im_start_positions[1]
            user_end = im_end_positions[1]
            segment_data.append(create_segment(
                tokens, attributions, user_start, user_end + 1,
                segment_type='user_prompt'
            ))
        
        # Assistant preamble (from im_start to think)
        if len(im_start_positions) >= 3 and think_start > 0:
            asst_start = im_start_positions[2]
            segment_data.append(create_segment(
                tokens, attributions, asst_start, think_start + 1,
                segment_type='assistant_preamble'
            ))
    
    # Now process the CoT content (after <think>)
    cot_start = think_start + 1 if think_start >= 0 else 0
    cot_end = think_end if think_end > 0 else len(tokens)
    
    # Apply sentence splitting only to CoT portion
    cot_segments = segment_cot_content(
        tokens[cot_start:cot_end],
        attributions[cot_start:cot_end],
        token_offset=cot_start
    )
    segment_data.extend(cot_segments)
    
    # Handle post-think content if exists (the final answer)
    if think_end > 0 and think_end < len(tokens) - 1:
        segment_data.append(create_segment(
            tokens, attributions, think_end, len(tokens),
            segment_type='final_answer'
        ))
    
    return segment_data


def create_segment(tokens, attributions, start_idx, end_idx, segment_type='cot_step'):
    """Create a segment dictionary from token range."""
    def decode_token(tok):
        return tok.replace('Ġ', ' ').replace('Ċ', '\n')
    
    seg_toks = tokens[start_idx:end_idx]
    seg_attrs = attributions[start_idx:end_idx]
    step_text = ''.join(decode_token(t) for t in seg_toks)
    
    if len(seg_attrs) > 0:
        seg_attrs_arr = np.array(seg_attrs)
        strength = float(np.sum(np.abs(seg_attrs_arr)))
        normalized_strength = strength / np.sqrt(len(seg_attrs)) if len(seg_attrs) > 0 else 0.0
        consistency_num = float(np.abs(np.sum(seg_attrs_arr)))
        consistency = consistency_num / strength if strength > 0 else 0.0
    else:
        strength = 0.0
        normalized_strength = 0.0
        consistency = 0.0
    
    return {
        'step_text': step_text.strip(),
        'tokens': seg_toks,
        'token_start': start_idx,
        'token_end': end_idx,
        'strength': strength,
        'normalized_strength_pre': normalized_strength,
        'consistency': consistency,
        'segment_type': segment_type
    }


def segment_cot_content(tokens, attributions, token_offset=0):
    """
    Apply sentence-level splitting to CoT content.
    This is the core logic from your original function.
    """
    def decode_token(tok):
        return tok.replace('Ġ', ' ').replace('Ċ', '\n')
    
    # Build character -> token index mapping
    char_to_token = []
    full_text = ""
    for idx, tok in enumerate(tokens):
        decoded = decode_token(tok)
        for _ in decoded:
            char_to_token.append(idx)
        full_text += decoded
    
    if not full_text.strip():
        return []
    
    # LaTeX and code-block aware splitting pattern
    # Matches: $...$, $...$, \[...\], \(...\), and code blocks (multiple lines starting with common code patterns)
    latex_pattern = r'(\$\$[\s\S]*?\$\$|\$[\s\S]*?\$|\\\[[\s\S]*?\\\]|\\\([\s\S]*?\\\))'
    
    # Code block pattern: captures blocks of lines that look like code
    # Matches sequences of lines containing draw(), common programming syntax, or indented blocks
    code_block_pattern = r'((?:^[ \t]*(?:draw|label|import|def|class|for|while|if|return|print)\s*\([^\n]*\n?)+)'
    
    # Combined pattern - check for code blocks first, then LaTeX
    combined_pattern = r'(' + code_block_pattern[1:-1] + r'|' + latex_pattern[1:-1] + r')'
    
    chunks = re.split(combined_pattern, full_text, flags=re.MULTILINE)
    
    # Track character positions
    char_pos = 0
    step_char_spans = []
    current_step = ""
    current_start = 0
    
    for chunk in chunks:
        chunk_start = char_pos
        chunk_end = char_pos + len(chunk)
        
        if re.match(latex_pattern, chunk):
            # LaTeX block - add to current step without splitting
            if current_step == "":
                current_start = chunk_start
            current_step += chunk
            char_pos = chunk_end
            continue
        
        # Text chunk - split by sentence delimiters
        # Also split on double newlines (paragraph breaks in reasoning)
        sub_parts = re.split(r'([.?!]\s+|\n\n+)', chunk)
        
        sub_pos = chunk_start
        for part in sub_parts:
            if current_step == "" and part.strip():
                current_start = sub_pos
            current_step += part
            sub_pos += len(part)
            
            # Split on sentence delimiters or paragraph breaks
            if re.match(r'[.?!]\s+|\n\n+', part):
                if len(current_step.strip()) > 5:
                    step_char_spans.append((current_start, sub_pos, current_step.strip()))
                    current_step = ""
        
        char_pos = chunk_end
    
    # Don't forget the last step
    if current_step.strip() and len(current_step.strip()) > 5:
        step_char_spans.append((current_start, char_pos, current_step.strip()))
    
    # Convert character spans to token spans
    segment_data = []
    for start_char, end_char, step_text in step_char_spans:
        if len(char_to_token) == 0:
            continue
            
        start_char = max(0, min(start_char, len(char_to_token) - 1))
        end_char = max(0, min(end_char - 1, len(char_to_token) - 1))
        
        start_token = char_to_token[start_char]
        end_token = char_to_token[end_char]
        
        seg_toks = tokens[start_token:end_token + 1]
        seg_attrs = attributions[start_token:end_token + 1]
        
        if len(seg_attrs) > 0:
            seg_attrs_arr = np.array(seg_attrs)
            strength = float(np.sum(np.abs(seg_attrs_arr)))
            normalized_strength = strength / np.sqrt(len(seg_attrs))
            consistency_num = float(np.abs(np.sum(seg_attrs_arr)))
            consistency = consistency_num / strength if strength > 0 else 0.0
            
            segment_data.append({
                'step_text': step_text,
                'tokens': seg_toks,
                'token_start': start_token + token_offset,
                'token_end': end_token + 1 + token_offset,
                'strength': strength,
                'normalized_strength_pre': normalized_strength,
                'consistency': consistency,
                'segment_type': 'cot_step'
            })
    
    return segment_data


def normalize_segment_strengths(segment_data: List[Dict]):
    """Normalize strengths across all segments."""
    total_strength = sum(item['normalized_strength_pre'] for item in segment_data)
    
    if total_strength > 0:
        for item in segment_data:
            item['normalized_strength_post'] = item['normalized_strength_pre'] / total_strength
    else:
        for item in segment_data:
            item['normalized_strength_post'] = 0.0
            
    return segment_data


# Example usage:
segment_data_pre = segment_tokens_with_cot_splitter(attr_result, include_prompt_segments=True)
segment_data_post = normalize_segment_strengths(segment_data_pre)
# 
# # Filter by segment type if needed:
cot_only = [s for s in segment_data_post if s['segment_type'] == 'cot_step']
prompt_segments = [s for s in segment_data_post if s['segment_type'] in ['system_prompt', 'user_prompt']]

Found 181 steps


In [7]:
for i, d in enumerate(segment_data_post):
    d['orig_idx'] = i
    
segments_sorted = sorted(
    segment_data_post,
    key=lambda d: d['normalized_strength_post'],
    reverse=True
)

len(segments_sorted)

79

In [8]:
TAU = 0.8
BETA = 0.8

k_star = 0
total_normalized_strength = 0
for item in segments_sorted:
    total_normalized_strength += item['normalized_strength_post']
    k_star += 1
    if total_normalized_strength > TAU:
        break
    
print(len(segments_sorted))
top_segments = segments_sorted[:k_star]
print(len(top_segments))

important_segments = []
for item in top_segments:
    if item['consistency'] < BETA:
        important_segments.append(item)
        
print(len(important_segments))

79
37
24


In [9]:
for seg in important_segments:
    print(seg['orig_idx'])

1
78
0
63
21
64
33
2
3
16
62
50
35
15
34
36
54
60
72
61
31
7
4
46


In [10]:
important_segments_orig_order = sorted(
    important_segments,
    key=lambda data: data['orig_idx'],
    reverse=False
)

for seg in important_segments_orig_order:
    print(seg['orig_idx'])

0
1
2
3
4
7
15
16
21
31
33
34
35
36
46
50
54
60
61
62
63
64
72
78


In [11]:
keep_indices = [seg['orig_idx'] for seg in important_segments_orig_order]
keep_indices = [idx for idx in keep_indices if idx <= essp_index]
print(len(keep_indices))

23


In [22]:
import numpy as np
import re
from typing import Dict, List, Tuple, Optional

class CoTSplitter:
    """
    Splits reasoning text into logical steps (sentences), 
    but protects LaTeX math environments from being split.
    """
    @staticmethod
    def split(text):
        pattern = r'(\$\$[\s\S]*?\$\$|\$[\s\S]*?\$|\\\[[\s\S]*?\\\]|\\\([\s\S]*?\\\))'
        chunks = re.split(pattern, text)
        
        steps = []
        current_step = ""
        
        for chunk in chunks:
            if re.match(pattern, chunk):
                current_step += chunk
                continue
            
            sub_parts = re.split(r'([.?!]\s+)', chunk)
            
            for part in sub_parts:
                current_step += part
                if re.match(r'[.?!]\s+', part):
                    if len(current_step.strip()) > 5:
                        steps.append(current_step.strip())
                        current_step = ""
        
        if current_step.strip():
            steps.append(current_step.strip())
            
        return steps


def decode_tokens(tokens: List[str]) -> str:
    """Decode tokenizer tokens to plain text."""
    return ''.join(tok.replace('Ġ', ' ').replace('Ċ', '\n') for tok in tokens)


def build_char_to_token_map(tokens: List[str]) -> List[int]:
    """Build mapping from character index to token index."""
    char_to_token = []
    for idx, tok in enumerate(tokens):
        decoded = tok.replace('Ġ', ' ').replace('Ċ', '\n')
        char_to_token.extend([idx] * len(decoded))
    return char_to_token


def find_step_in_text(step_text: str, full_text: str, search_start: int = 0) -> Tuple[int, int]:
    """
    Find the character span of a step in the full text.
    Returns (start_char, end_char) or (-1, -1) if not found.
    """
    # Try exact match first
    idx = full_text.find(step_text, search_start)
    if idx != -1:
        return idx, idx + len(step_text)
    
    # Try normalized match (collapse whitespace)
    step_normalized = ' '.join(step_text.split())
    text_normalized = ' '.join(full_text[search_start:].split())
    idx_norm = text_normalized.find(step_normalized)
    if idx_norm != -1:
        # Map back to original position (approximate)
        return search_start + idx_norm, search_start + idx_norm + len(step_text)
    
    return -1, -1


def map_steps_to_tokens(
    steps: List[str],
    tokens: List[str],
    attributions: List[float]
) -> List[Dict]:
    """
    Map CoTSplitter steps to their corresponding tokens and attributions.
    
    Args:
        steps: List of step strings from CoTSplitter.split()
        tokens: List of tokens from the tokenizer
        attributions: List of attribution scores per token
    
    Returns:
        List of segment dictionaries with token spans and attribution data
    """
    full_text = decode_tokens(tokens)
    char_to_token = build_char_to_token_map(tokens)
    
    segments = []
    search_start = 0
    
    for step_idx, step_text in enumerate(steps):
        # Find this step in the full text
        char_start, char_end = find_step_in_text(step_text, full_text, search_start)
        
        if char_start == -1:
            # Step not found - this shouldn't happen if steps came from same text
            print(f"Warning: Step {step_idx} not found in text: {step_text[:50]}...")
            segments.append({
                'step_idx': step_idx,
                'step_text': step_text,
                'tokens': [],
                'token_start': None,
                'token_end': None,
                'attributions': [],
                'strength': 0.0,
                'normalized_strength': 0.0,
                'consistency': 0.0,
                'found': False
            })
            continue
        
        # Update search position for next step
        search_start = char_end
        
        # Map character positions to token indices
        # Clamp to valid range
        char_start_clamped = max(0, min(char_start, len(char_to_token) - 1))
        char_end_clamped = max(0, min(char_end - 1, len(char_to_token) - 1))
        
        token_start = char_to_token[char_start_clamped]
        token_end = char_to_token[char_end_clamped] + 1  # Exclusive end
        
        # Extract tokens and attributions for this segment
        seg_tokens = tokens[token_start:token_end]
        seg_attrs = attributions[token_start:token_end]
        
        # Compute attribution statistics
        if len(seg_attrs) > 0:
            seg_attrs_arr = np.array(seg_attrs)
            strength = float(np.sum(np.abs(seg_attrs_arr)))
            normalized_strength = strength / np.sqrt(len(seg_attrs))
            consistency_num = float(np.abs(np.sum(seg_attrs_arr)))
            consistency = consistency_num / strength if strength > 0 else 0.0
        else:
            strength = 0.0
            normalized_strength = 0.0
            consistency = 0.0
        
        segments.append({
            'step_idx': step_idx,
            'step_text': step_text,
            'tokens': seg_tokens,
            'token_start': token_start,
            'token_end': token_end,
            'attributions': seg_attrs,
            'strength': strength,
            'normalized_strength': normalized_strength,
            'consistency': consistency,
            'found': True
        })
    
    return segments


def get_segments_by_indices(
    segments: List[Dict],
    indices: List[int]
) -> List[Dict]:
    """Get specific segments by their step indices."""
    return [seg for seg in segments if seg['step_idx'] in indices]


def reconstruct_pruned_cot(
    segments: List[Dict],
    keep_indices: List[int],
    join_str: str = " "
) -> str:
    """
    Reconstruct CoT text keeping only specified segment indices.
    
    Args:
        segments: Full list of segment dictionaries
        keep_indices: List of step indices to keep
        join_str: String to join segments with
    
    Returns:
        Pruned CoT text
    """
    kept = [seg['step_text'] for seg in segments if seg['step_idx'] in keep_indices]
    return join_str.join(kept)


def get_pruned_tokens_and_attributions(
    segments: List[Dict],
    keep_indices: List[int]
) -> Tuple[List[str], List[float]]:
    """
    Get concatenated tokens and attributions for kept segments.
    
    Args:
        segments: Full list of segment dictionaries
        keep_indices: List of step indices to keep
    
    Returns:
        (tokens, attributions) for the pruned CoT
    """
    kept_tokens = []
    kept_attrs = []
    
    for seg in segments:
        if seg['step_idx'] in keep_indices and seg['found']:
            kept_tokens.extend(seg['tokens'])
            kept_attrs.extend(seg['attributions'])
    
    return kept_tokens, kept_attrs


def normalize_segment_strengths(segments: List[Dict]) -> List[Dict]:
    """Normalize strengths across all segments to sum to 1."""
    total = sum(seg['normalized_strength'] for seg in segments)
    
    if total > 0:
        for seg in segments:
            seg['normalized_strength_global'] = seg['normalized_strength'] / total
    else:
        for seg in segments:
            seg['normalized_strength_global'] = 0.0
    
    return segments


# =============================================================================
# EXAMPLE USAGE
# =============================================================================
#
# # 1. Get your CoT text and tokenized data
cot_text = cot_up_to_hsp1  # The raw CoT text
tokens = tokenizer(cot_text)    # From tokenizer
attributions = attr_result  # From integrated gradients

full_text = decode_tokens(tokens)
steps = CoTSplitter.split(full_text)
segments = map_steps_to_tokens(steps, tokens, attributions)
#
# # 2. Split with CoTSplitter (canonical segmentation)
steps = CoTSplitter.split(cot_text)
print(f"Found {len(steps)} steps")
#
# # 3. Map steps to tokens
segments = map_steps_to_tokens(steps, tokens, attributions)
segments
#
# # 4. Normalize strengths
# segments = normalize_segment_strengths(segments)
#
# # 5. Your pruning logic determines which indices to keep
# keep_indices = [0, 3, 7, 12, 43]  # Example: from your importance analysis
#
# # 6. Reconstruct pruned CoT
# pruned_text = reconstruct_pruned_cot(segments, keep_indices)
# pruned_tokens, pruned_attrs = get_pruned_tokens_and_attributions(segments, keep_indices)
#
# # 7. Access specific segment data
# for idx in keep_indices:
#     seg = segments[idx]
#     print(f"Step {idx}: strength={seg['strength']:.3f}")
#     print(f"  Text: {seg['step_text'][:80]}...")
#     print(f"  Tokens: {seg['token_start']} to {seg['token_end']}")

KeyError: slice(0, 2, None)

## Build the Prompt

In [109]:
problem_prompt = problem1['problem']

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

base_prompt = build_base_prompt(problem_prompt)

#prompt = f"{base_prompt}<think>\n{pruned_cot_up_to_essp}"
prompt = f"{base_prompt}<think>\n{pruned_cot_up_to_essp}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"


## Run the Model

In [110]:
outputs = model.generate(
    [prompt], params_handoff, use_tqdm=True
)

Processed prompts: 100%|██████████| 20/20 [00:00<00:00, 44.33it/s, est. speed input: 56949.32 toks/s, output: 209.42 toks/s]


In [111]:
# Get generation results
output = outputs[0]
completions = [o.text for o in output.outputs]
print(completions)

['93}.', '183}.', '122}.', '93}.', '18}', '243}.', '243}.', '243}.', '243}.', '183}.', '183}.', '189}.', '243}.', '18}.', '189}.', '93}.', '32}.', '183}.', '243}.', '243}.']


In [112]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

ground_truth = problem1['solution']

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth, is_partial=True)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '93'
  Norm Truth: '48'

  Norm Pred: '183'
  Norm Truth: '48'

  Norm Pred: '122'
  Norm Truth: '48'

  Norm Pred: '93'
  Norm Truth: '48'

  Norm Pred: '18'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '183'
  Norm Truth: '48'

  Norm Pred: '183'
  Norm Truth: '48'

  Norm Pred: '189'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '18'
  Norm Truth: '48'

  Norm Pred: '189'
  Norm Truth: '48'

  Norm Pred: '93'
  Norm Truth: '48'

  Norm Pred: '32'
  Norm Truth: '48'

  Norm Pred: '183'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'


0.0

## Testing Anti-Pruned ESSP

In [114]:
keep_indices = [seg['orig_idx'] for seg in important_segments_orig_order]
drop_indices = [i for i in range(len(segment_data_post)) if i not in keep_indices]
drop_indices = [idx for idx in drop_indices if idx <= essp_index]
drop_indices

essp_anti_pruned_steps = [step for idx, step in enumerate(problem1['steps']) if idx in drop_indices]
anti_pruned_cot_up_to_essp = "\n".join(essp_anti_pruned_steps)
print(anti_pruned_cot_up_to_essp)

Let me start by understanding the pattern from the given figures.
First, let me look at Figure 1.
Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).
So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.
The Asymptote code draws a more complex figure.
Let me try to visualize it.
Let me count the endpoints in Figure 2.
Then there are lines from (4,1) to (5,0) to (6,1).
Wait, maybe it's better to think recursively.
Each endpoint from the previous figure is replaced by a Y.
In Figure 1, there are 3 endpoints.
Each of these endpoints is replaced by a Y in Figure 2.
A Y has three endpoints, but one of them is connected to the previous segment.
Wait, but when you replace an endpoint with a Y, how does that work?
Let me think.
Because the Y has two new branches.
But if you're replacing an endpoint, you would be adding two new se

In [ ]:
## Build the Prompt

In [115]:
problem_prompt = problem1['problem']

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

base_prompt = build_base_prompt(problem_prompt)

#prompt = f"{base_prompt}<think>\n{pruned_cot_up_to_essp}"
prompt = f"{base_prompt}<think>\n{anti_pruned_cot_up_to_essp}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"


In [120]:
anti_pruned_cot_up_to_essp

"Let me start by understanding the pattern from the given figures.\nFirst, let me look at Figure 1.\nWait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).\nSo, it's like a central point with three lines coming out: one straight down, and two going up to the sides.\nThe Asymptote code draws a more complex figure.\nLet me try to visualize it.\nLet me count the endpoints in Figure 2.\nThen there are lines from (4,1) to (5,0) to (6,1).\nWait, maybe it's better to think recursively.\nEach endpoint from the previous figure is replaced by a Y.\nIn Figure 1, there are 3 endpoints.\nEach of these endpoints is replaced by a Y in Figure 2.\nA Y has three endpoints, but one of them is connected to the previous segment.\nWait, but when you replace an endpoint with a Y, how does that work?\nLet me think.\nBecause the Y has two new branches.\nBut if you're replacing an endpoint, you would be 

## Run the Model

In [116]:
outputs = model.generate(
    [prompt], params_handoff, use_tqdm=True
)

Processed prompts: 100%|██████████| 20/20 [00:01<00:00, 15.49it/s, est. speed input: 22557.98 toks/s, output: 98.00 toks/s]


In [117]:
# Get generation results
output = outputs[0]
completions = [o.text for o in output.outputs]
print(completions)

['48}.', '48}.', '48}.', '48}.\n$$\n\nTo determine the number of endpoints in **Figure 5**, we analyze the', '48}.', '48}.', '48}.\n$$\\boxed{48}$$', '48}.', '48}.', '48}.\nLet me verify this with the pattern.\n\nFigure 1: 3 endpoints.\nFigure', '48}.', '48}.', '48}.', '48}.', '48}.', '48}.', '48}.', '48}.', '48}.\n\\boxed{48}', '48}.']


In [118]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

ground_truth = problem1['solution']

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth, is_partial=True)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'

  Norm Pred: '48'
  Norm Truth: '48'


1.0

## Testing Anti-Pruned ESSP (same length as pruned (random selection))

In [129]:
import random

random.seed(9001) 

keep_indices = [seg['orig_idx'] for seg in important_segments_orig_order]
keep_indices = [idx for idx in keep_indices if idx <= essp_index]
print(len(keep_indices))
drop_indices = [i for i in range(len(segment_data_post)) if i not in keep_indices]
drop_indices = [idx for idx in drop_indices if idx <= essp_index]
drop_indices_trimmed = sorted(random.sample(drop_indices, len(keep_indices)))
print(len(keep_indices))
print(len(drop_indices))
print(len(drop_indices_trimmed))

essp_anti_pruned_steps_trimmed = [step for idx, step in enumerate(problem1['steps']) if idx in drop_indices_trimmed]
anti_pruned_cot_up_to_essp_trimmed = "\n".join(essp_anti_pruned_steps_trimmed)
print(anti_pruned_cot_up_to_essp_trimmed)

24
24
49
24
Wait, actually, looking at the Asymptote code, Figure 1 is drawn with three line segments: from (0,0) down to (0,-3), then from (0,0) to (-2,2) and (0,0) to (2,2).
So, it's like a central point with three lines coming out: one straight down, and two going up to the sides.
The Asymptote code draws a more complex figure.
Then there are lines from (4,1) to (5,0) to (6,1).
A Y has three endpoints, but one of them is connected to the previous segment.
Let me think.
But if you're replacing an endpoint, you would be adding two new segments branching off from that endpoint, thereby replacing the single endpoint with two new endpoints.
Wait, but in Figure 1, there are three endpoints.
If each endpoint is replaced by a Y, then each of those three endpoints becomes two endpoints.
But let me check with the Asymptote code.
So, the bottom endpoints are (4,-3) and (6,-3).
Then from (4,1) goes to (3,1) and (4,2).
Additionally, from (3,1) there's a line to (8.5,1)?
- Then, from (5,0), there

## Build the Prompt

In [130]:
problem_prompt = problem1['problem']

SYSTEM_PROMPT = (
    "You are a helpful reasoning assistant. "
    "Output your final answer inside \\boxed{}."
)

def build_base_prompt(problem):
    """Build the base prompt for a problem."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": problem},
    ]
    return tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

base_prompt = build_base_prompt(problem_prompt)

#prompt = f"{base_prompt}<think>\n{pruned_cot_up_to_essp}"
prompt = f"{base_prompt}<think>\n{anti_pruned_cot_up_to_essp_trimmed}"
prompt += "\n</think>\nTherefore, the final answer is \\boxed{"


## Run the Model

In [131]:
outputs = model.generate(
    [prompt], params_handoff, use_tqdm=True
)

Processed prompts: 100%|██████████| 20/20 [00:00<00:00, 40.76it/s, est. speed input: 44516.35 toks/s, output: 202.15 toks/s]


In [132]:
# Get generation results
output = outputs[0]
completions = [o.text for o in output.outputs]
print(completions)

['243}.', '243}.', '162}.', '243}.', '243}.', '243}.', '243}.', '243}.', '243}.', '243}.', '243}.', '243}.', '243}.', '16}.', '162}.', '243}.', '243}.', '243}.', '243}.', '243}.']


In [133]:
import re

class MathVerifier:
    """Answer extractor with normalization."""
    
    @staticmethod
    def extract_answer(text):
        if not text: return None
        # Handle cases where whitespace is inserted like \boxed { content }
        start_indices = [m.start() for m in re.finditer(r'\\boxed\s*\{', text)]
        if not start_indices: return None
        
        for start in start_indices:
            balance = 0
            # Find the first '{' after the \boxed
            content_start = text.find('{', start) + 1
            for i in range(content_start, len(text)):
                char = text[i]
                if char == '{': balance += 1
                elif char == '}':
                    if balance == 0: return text[content_start:i]
                    balance -= 1
        return None

    @staticmethod
    def extract_from_partial(text):
        """
        Extracts content when the prompt ends with \boxed{
        We look for the first closing brace '}' that isn't balanced by an opening '{'
        within the generated text itself.
        """
        if not text: return None
        balance = 0
        for i, char in enumerate(text):
            if char == '{':
                balance += 1
            elif char == '}':
                # If balance is 0, this '}' closes the ghost \boxed{ from the prompt
                if balance == 0:
                    return text[:i]
                balance -= 1
        # Fallback: if no closing brace found, return everything (model stopped early)
        return text

    @staticmethod
    def normalize_answer(text):
        if text is None: return ""
        text = text.strip().replace(" ", "")
        
        # --- FIX 1: Handle JSON-style double escaping ---
        # Turns \\left into \left, \\dfrac into \dfrac
        text = text.replace("\\\\", "\\") 
        
        # --- FIX 2: Standardize LaTeX fractions ---
        # Handle all fraction formats: \dfrac, \tfrac, dfrac (missing backslash)
        text = text.replace(r"\dfrac", r"\frac").replace(r"\tfrac", r"\frac")
        text = text.replace("dfrac", "frac")  # Handle cases where backslash was stripped
        
        # --- FIX 3: Remove sizing commands ---
        text = text.replace(r"\left", "").replace(r"\right", "")
        
        # --- FIX 4: Remove \text{...} wrappers ---
        text = re.sub(r'\\text\{(.*?)\}', r'\1', text)
        
        # --- FIX 5: Normalize plain fractions to LaTeX format ---
        # Convert "7/20" to "frac{7}{20}" for consistent comparison
        # This handles simple cases like "7/20", "12/5", etc.
        text = re.sub(r'(\d+)/(\d+)', r'frac{\1}{\2}', text)
        
        return text

    @staticmethod
    def is_correct(generated_text, ground_truth, is_partial=False):
        if is_partial:
            pred = MathVerifier.extract_from_partial(generated_text)
        else:
            pred = MathVerifier.extract_answer(generated_text)
            
        truth = MathVerifier.extract_answer(ground_truth)
        if truth is None: truth = ground_truth
        
        # Debug print to verify the fix works in your logs
        print()
        print(f"  Norm Pred: '{MathVerifier.normalize_answer(pred)}'")
        print(f"  Norm Truth: '{MathVerifier.normalize_answer(truth)}'")

        return MathVerifier.normalize_answer(pred) == MathVerifier.normalize_answer(truth)

ground_truth = problem1['solution']

correct_count = sum(
    1
    for comp in completions
    if MathVerifier.is_correct(comp, ground_truth, is_partial=True)
)
accuracy = correct_count / 20
accuracy


  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '162'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '16'
  Norm Truth: '48'

  Norm Pred: '162'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'

  Norm Pred: '243'
  Norm Truth: '48'


0.0